# Signal Lab

A miniature scientific workflow: synthesize a sensor signal, clean it, inspect the frequency domain, and finish with a spectrogram. It is deterministic, offline, and designed to create useful image outputs quickly.


In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
from IPython.display import display

rng = np.random.default_rng(11)
fs = 240
duration = 6.0
time = np.arange(0, duration, 1 / fs)
carrier = 0.9 * np.sin(2 * np.pi * 8 * time)
harmonic = 0.34 * np.sin(2 * np.pi * 21 * time + 0.4)
drift = 0.22 * np.sin(2 * np.pi * 0.35 * time)
noise = rng.normal(0, 0.18, size=time.size)
raw = carrier + harmonic + drift + noise
b, a = signal.butter(4, [4, 32], btype="bandpass", fs=fs)
clean = signal.filtfilt(b, a, raw)

metrics = pd.DataFrame(
    {
        "series": ["raw", "filtered"],
        "mean": [raw.mean(), clean.mean()],
        "std": [raw.std(), clean.std()],
        "peak_to_peak": [np.ptp(raw), np.ptp(clean)],
    }
).round(3)
display(metrics)


## Time-domain inspection

The first plot gives readers the result immediately. The code remains available in a collapsible block underneath the output.


In [ ]:
import matplotlib.pyplot as plt

window = slice(0, int(fs * 2.2))
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(time[window], raw[window], color="#94a3b8", linewidth=1.1, label="raw sensor")
ax.plot(time[window], clean[window], color="#087f5b", linewidth=2.0, label="bandpass filtered")
ax.set_title("Two seconds of sensor data")
ax.set_xlabel("seconds")
ax.set_ylabel("amplitude")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()


## Frequency-domain summary

Frequency plots make a good stress test for media-heavy reports: they are visual, dense, and much nicer to read with an outline.


In [ ]:
freq, power = signal.welch(clean, fs=fs, nperseg=512)
peaks, _ = signal.find_peaks(power, prominence=0.01)
top = pd.DataFrame({"frequency_hz": freq[peaks], "power": power[peaks]}).sort_values("power", ascending=False).head(5)
display(top.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(freq, power, color="#1d4ed8", linewidth=2)
ax.scatter(top["frequency_hz"], top["power"], color="#b45309", zorder=3, label="top peaks")
ax.set_xlim(0, 48)
ax.set_title("Welch power spectral density")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("power")
ax.grid(True, which="both", alpha=0.25)
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()


## Spectrogram

A final heatmap shows that the viewer handles larger chart images as lazy assets instead of embedding the media into section JSON.


In [ ]:
f, t, sxx = signal.spectrogram(clean, fs=fs, nperseg=160, noverlap=120)
fig, ax = plt.subplots(figsize=(10, 4.4))
mesh = ax.pcolormesh(t, f, 10 * np.log10(sxx + 1e-8), shading="gouraud", cmap="viridis")
ax.set_ylim(0, 45)
ax.set_title("Spectrogram of filtered signal")
ax.set_xlabel("seconds")
ax.set_ylabel("frequency (Hz)")
fig.colorbar(mesh, ax=ax, label="power (dB)")
fig.tight_layout()
plt.show()
